# 01. Data Quality Assessment & Remediation (Credit Applications)

## Scope
We assess data quality on the `raw_credit_applications.json` dataset and produce:
- A structured **issue inventory** (counts + percentages) mapped to quality dimensions
- A **cleaned dataset** with documented remediation steps
- Before/after comparison tables exported as CSV for reporting

## Quality dimensions used
- **Completeness**: missing / incomplete values
- **Consistency**: inconsistent formats or encodings
- **Validity**: invalid values, impossible ranges, unparsable dates
- **Accuracy (proxy checks)**: anomalies that strongly suggest incorrectness (e.g., negative months)

> Note: This notebook follows an engineering-style workflow: **load → profile → detect issues → remediate → validate → export artifacts**.

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import re
from dateutil import parser as date_parser

RAW_PATH = Path("raw_credit_applications.json")
assert RAW_PATH.exists(), f"File not found: {RAW_PATH.resolve()}"

records = json.loads(RAW_PATH.read_text(encoding="utf-8"))
len(records), type(records)

(502, list)

## 2. JSON Normalization and Tabular Structuring

The raw dataset contains nested JSON structures (e.g., `applicant_info`, `financials`, `decision`, and `spending_behavior`).  
In order to perform systematic data quality checks, we must first transform this semi-structured data into a flat, tabular format.

In this step we:

- Extract nested fields into a single structured row per application.
- Standardize category values using a normalization function (`normalize_category`).
- Create engineered aggregate features from transaction-level data:
  - `spend_total_amount`: total spending amount across transactions.
  - `spend_items_count`: number of recorded spending events.
- Ensure robustness by handling missing or malformed nested structures safely.

The result is a clean DataFrame (`df`) where each row represents one credit application, enabling column-level and record-level quality validation.

In [2]:
def normalize_category(c):
    if c is None: return "unknown"
    if not isinstance(c, str): return "unknown"
    x = re.sub(r"\s+", " ", c.strip().lower())
    mapping = {"grocery":"groceries"}
    return mapping.get(x, x.replace(" ", "_"))

rows = []
for r in records:
    applicant = r.get("applicant_info", {}) or {}
    financials = r.get("financials", {}) or {}
    decision = r.get("decision", {}) or {}
    spend = r.get("spending_behavior", []) or []

    row = {
        "_id": r.get("_id"),
        "processing_timestamp": r.get("processing_timestamp"),

        "full_name": applicant.get("full_name"),
        "email": applicant.get("email"),
        "ssn": applicant.get("ssn"),
        "ip_address": applicant.get("ip_address"),
        "gender": applicant.get("gender"),
        "date_of_birth": applicant.get("date_of_birth"),
        "zip_code": applicant.get("zip_code"),

        "annual_income": financials.get("annual_income"),
        "credit_history_months": financials.get("credit_history_months"),
        "debt_to_income": financials.get("debt_to_income"),
        "savings_balance": financials.get("savings_balance"),

        "loan_approved": decision.get("loan_approved"),
        "rejection_reason": decision.get("rejection_reason"),
    }

    # spending aggregates
    if isinstance(spend, list):
        amounts = pd.to_numeric(pd.Series([x.get("amount") for x in spend if isinstance(x, dict)]), errors="coerce")
        row["spend_total_amount"] = float(amounts.sum()) if len(amounts) else 0.0
        row["spend_items_count"] = len(spend)
    else:
        row["spend_total_amount"] = np.nan
        row["spend_items_count"] = np.nan

    rows.append(row)

df = pd.DataFrame(rows)
df.head()

,_id,processing_timestamp,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,rejection_reason,spend_total_amount,spend_items_count
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,1517.0,3
1,app_037,None,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,947.0,3
2,app_215,None,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,None,109.0,1
3,app_024,None,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,None,575.0,1
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,463.0,1


## 3. Issue Logging Utilities (Reusable Metrics)

Before detecting specific problems, we define reusable helper functions to ensure that every issue is reported consistently.

- `count_pct(mask)`: converts a boolean mask into a **count** and **percentage**.
- `show_issue(name, dimension, mask)`: formats results into a standardized dictionary with:
  - issue name
  - quality dimension (Completeness / Consistency / Validity / Accuracy)
  - count and percentage

This structure supports an engineering-style workflow where issues are measured in a repeatable, auditable way.

In [3]:
def count_pct(mask: pd.Series) -> tuple[int, float]:
    c = int(mask.sum())
    pct = round(c / len(mask) * 100, 2) if len(mask) else 0.0
    return c, pct

def show_issue(name, dimension, mask):
    c, p = count_pct(mask)
    return {"issue": name, "dimension": dimension, "count": c, "pct": p}

issues = []
n = len(df)
n

502

## 4. Consistency Check: Duplicate Identifiers

We check whether the unique identifier (`_id`) contains duplicates.  
Duplicate IDs indicate **consistency problems** because the same application may be represented multiple times, leading to inflated counts and biased analytics.

We flag all records that share an `_id` with at least one other record.

In [4]:
dup_mask = df["_id"].duplicated(keep=False)
issues.append(show_issue("Duplicate _id", "Consistency", dup_mask))
pd.DataFrame(issues).tail(1)

,issue,dimension,count,pct
0,Duplicate _id,Consistency,4,0.8


## 5. Completeness Checks: Missing Values and Incomplete Records

We evaluate completeness on a set of **key fields** required for credit decisioning and analysis (identity, demographics, financial indicators, and outcome).

Two complementary checks are performed:

1. **Column-level missingness**: percentage of missing values per key field (helps identify problematic attributes).
2. **Record-level incompleteness**: rows where **at least one key field is missing**, indicating an incomplete application record.

These metrics quantify how much of the dataset is usable without imputation or enrichment.

In [5]:
key_fields = ["_id","full_name","gender","date_of_birth","zip_code",
              "annual_income","credit_history_months","debt_to_income","savings_balance",
              "loan_approved"]

missing_stats = (
    df[key_fields].isna().mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_stats.columns = ["column","missing_pct"]
missing_stats["missing_pct"] = (missing_stats["missing_pct"]*100).round(2)
missing_stats

,column,missing_pct
0,annual_income,1.0
1,gender,0.2
2,date_of_birth,0.2
3,zip_code,0.2
4,_id,0.0
5,full_name,0.0
6,credit_history_months,0.0
7,debt_to_income,0.0
8,savings_balance,0.0
9,loan_approved,0.0


In [6]:
incomplete_mask = df[key_fields].isna().any(axis=1)
issues.append(show_issue("Incomplete record (≥1 key field missing)", "Completeness", incomplete_mask))
pd.DataFrame(issues).tail(1)

,issue,dimension,count,pct
1,Incomplete record (≥1 key field missing),Completeness,6,1.2


In [7]:
def type_counts(col):
    return df[col].map(lambda x: type(x).__name__).value_counts()

for col in ["annual_income","credit_history_months","debt_to_income","savings_balance"]:
    print(col)
    display(type_counts(col).to_frame("count"))

annual_income


,count
annual_income,
int,488
str,8
NoneType,5
float,1


credit_history_months


,count
credit_history_months,
int,502


debt_to_income


,count
debt_to_income,
float,502


savings_balance


,count
savings_balance,
int,502


In [8]:
for col in ["annual_income","credit_history_months","debt_to_income","savings_balance"]:
    numeric = pd.to_numeric(df[col], errors="coerce")
    mask = df[col].notna() & numeric.isna()
    issues.append(show_issue(f"{col} stored as non-numeric", "Consistency", mask))

pd.DataFrame(issues)

,issue,dimension,count,pct
0,Duplicate _id,Consistency,4,0.8
1,Incomplete record (≥1 key field missing),Completeness,6,1.2
2,annual_income stored as non-numeric,Consistency,0,0.0
3,credit_history_months stored as non-numeric,Consistency,0,0.0
4,debt_to_income stored as non-numeric,Consistency,0,0.0
5,savings_balance stored as non-numeric,Consistency,0,0.0


## 6. Consistency Check: Gender Encoding Standardization

We inspect the raw values of `gender` to understand how the attribute is encoded in practice (e.g., `male/female` vs `m/f`).

We then define an allowed set of standard encodings and flag all non-null values that do not belong to it.  
This identifies **inconsistent categorical coding**, which can break downstream grouping/segmentation and bias reporting.

Output:
- list of observed raw categories (`gender_values`)
- issue log entry: **Gender non-standard coding** (Consistency)

In [9]:
gender_raw = df["gender"].dropna().astype(str).str.strip()
gender_values = sorted(gender_raw.unique().tolist())
gender_values

['', 'F', 'Female', 'M', 'Male']

In [10]:
allowed = {"male","female","m","f"}
gender_norm = gender_raw.str.lower()
nonstandard_mask = df["gender"].notna() & ~gender_norm.isin(allowed)
issues.append(show_issue("Gender non-standard coding", "Consistency", nonstandard_mask))
pd.DataFrame(issues).tail(1)

,issue,dimension,count,pct
6,Gender non-standard coding,Consistency,2,0.4


## 7. Validity Check: Date of Birth Parsing

To validate `date_of_birth`, we attempt to parse the field into a proper date type.

- We treat empty strings and non-string values as invalid (`NaT`).
- We use `dayfirst=True` to handle common European formats (e.g., `31/12/2020`).
- Records where the original value is not null but parsing fails are flagged as **invalid/unparseable dates**.

This step captures **validity issues** that affect derived attributes (e.g., age) and any time-based analysis.

In [11]:
def parse_date(x):
    if x is None or (isinstance(x, str) and x.strip()==""):
        return pd.NaT
    if not isinstance(x, str):
        return pd.NaT
    try:
        # dayfirst=True per gestire formati tipo 31/12/2020
        return pd.Timestamp(date_parser.parse(x, dayfirst=True).date())
    except Exception:
        return pd.NaT

dob_parsed = df["date_of_birth"].apply(parse_date)
invalid_dob_mask = dob_parsed.isna() & df["date_of_birth"].notna()
issues.append(show_issue("Invalid/unparseable date_of_birth", "Validity", invalid_dob_mask))
pd.DataFrame(issues).tail(1)

,issue,dimension,count,pct
7,Invalid/unparseable date_of_birth,Validity,4,0.8


## 8. Validity Checks: Numeric Range Constraints

In this step we verify that key financial indicators fall within logically valid ranges.

The following rules are applied:

- `credit_history_months` must be non-negative.
- `annual_income` must be non-negative.
- `savings_balance` must be non-negative.
- `debt_to_income` must fall within a reasonable range (0 to 1.5).

Negative values in financial attributes indicate clear data validity violations.  
For `debt_to_income`, values outside the defined threshold may indicate incorrect scaling (e.g., percentage vs ratio) or data entry errors.

Each violation is logged in the issue inventory with counts and percentages.

In [12]:
ch = pd.to_numeric(df["credit_history_months"], errors="coerce")
neg_ch_mask = ch < 0
issues.append(show_issue("Negative credit_history_months", "Validity", neg_ch_mask))

inc = pd.to_numeric(df["annual_income"], errors="coerce")
neg_inc_mask = inc < 0
issues.append(show_issue("Negative annual_income", "Validity", neg_inc_mask))

sav = pd.to_numeric(df["savings_balance"], errors="coerce")
neg_sav_mask = sav < 0
issues.append(show_issue("Negative savings_balance", "Validity", neg_sav_mask))

dti = pd.to_numeric(df["debt_to_income"], errors="coerce")
dti_out_mask = (dti < 0) | (dti > 1.5)  # soglia ragionevole, puoi motivarla
issues.append(show_issue("Debt-to-income out of range (<0 or >1.5)", "Validity", dti_out_mask))

pd.DataFrame(issues)

,issue,dimension,count,pct
0,Duplicate _id,Consistency,4,0.8
1,Incomplete record (≥1 key field missing),Completeness,6,1.2
2,annual_income stored as non-numeric,Consistency,0,0.0
3,credit_history_months stored as non-numeric,Consistency,0,0.0
4,debt_to_income stored as non-numeric,Consistency,0,0.0
5,savings_balance stored as non-numeric,Consistency,0,0.0
6,Gender non-standard coding,Consistency,2,0.4
7,Invalid/unparseable date_of_birth,Validity,4,0.8
8,Negative credit_history_months,Validity,2,0.4
9,Negative annual_income,Validity,0,0.0


In [13]:
age_years = (pd.Timestamp.today().normalize() - dob_parsed).dt.days / 365.25
implausible_age_mask = (age_years < 18) | (age_years > 100)
implausible_age_mask = implausible_age_mask.fillna(False)
issues.append(show_issue("Implausible age (<18 or >100)", "Accuracy", implausible_age_mask))

pd.DataFrame(issues).sort_values(["pct","count"], ascending=False)

,issue,dimension,count,pct
1,Incomplete record (≥1 key field missing),Completeness,6,1.2
0,Duplicate _id,Consistency,4,0.8
7,Invalid/unparseable date_of_birth,Validity,4,0.8
6,Gender non-standard coding,Consistency,2,0.4
8,Negative credit_history_months,Validity,2,0.4
10,Negative savings_balance,Validity,1,0.2
11,Debt-to-income out of range (<0 or >1.5),Validity,1,0.2
2,annual_income stored as non-numeric,Consistency,0,0.0
3,credit_history_months stored as non-numeric,Consistency,0,0.0
4,debt_to_income stored as non-numeric,Consistency,0,0.0


## 9. Data Remediation and Cleaning

After identifying data quality issues, we apply controlled remediation steps to produce a cleaned dataset (`df_clean`).

Actions performed:

1. **Standardize categorical values**
   - Normalize `gender` into consistent labels ("Male", "Female", "Other/Unknown").

2. **Parse and derive attributes**
   - Convert `date_of_birth` into a parsed date.
   - Compute `age_years` as a derived analytical feature.

3. **Enforce numeric types**
   - Cast financial variables to numeric format with coercion.

4. **Handle invalid values transparently**
   - Negative or out-of-range values are set to `NaN`.
   - Flag columns are created to preserve traceability of corrections.

5. **Duplicate policy**
   - Remove duplicate records based on `_id` (keeping the first occurrence).

These steps ensure that remediation is transparent, auditable, and reproducible.

In [14]:
df_clean = df.copy()

# 1) Standardize gender
def normalize_gender(x):
    if x is None: return None
    if not isinstance(x, str): return x
    t = x.strip().lower()
    if t in {"m","male"}: return "Male"
    if t in {"f","female"}: return "Female"
    if t == "": return None
    return "Other/Unknown"

df_clean["gender"] = df_clean["gender"].apply(normalize_gender)

# 2) Parse DOB and compute age
df_clean["dob_parsed"] = df_clean["date_of_birth"].apply(parse_date)
df_clean["age_years"] = ((pd.Timestamp.today().normalize() - df_clean["dob_parsed"]).dt.days / 365.25)

# 3) Cast numerics
for col in ["annual_income","credit_history_months","debt_to_income","savings_balance"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# 4) Fix invalid values by setting to NaN + flags (trasparente e difendibile)
df_clean["flag_neg_credit_history_months"] = df_clean["credit_history_months"] < 0
df_clean.loc[df_clean["flag_neg_credit_history_months"], "credit_history_months"] = np.nan

df_clean["flag_neg_savings_balance"] = df_clean["savings_balance"] < 0
df_clean.loc[df_clean["flag_neg_savings_balance"], "savings_balance"] = np.nan

df_clean["flag_dti_out_of_range"] = (df_clean["debt_to_income"] < 0) | (df_clean["debt_to_income"] > 1.5)
df_clean.loc[df_clean["flag_dti_out_of_range"], "debt_to_income"] = np.nan

# 5) Remove duplicates (keep first) OR keep latest by timestamp (scegli una policy)
df_clean = df_clean.drop_duplicates(subset=["_id"], keep="first")

df_clean.head()

,_id,processing_timestamp,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,...,savings_balance,loan_approved,rejection_reason,spend_total_amount,spend_items_count,dob_parsed,age_years,flag_neg_credit_history_months,flag_neg_savings_balance,flag_dti_out_of_range
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,...,31212.0,False,algorithm_risk_score,1517.0,3,2001-09-03,24.487337,False,False,False
1,app_037,None,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,...,17915.0,False,algorithm_risk_score,947.0,3,1992-03-31,33.913758,False,False,False
2,app_215,None,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,...,37909.0,True,None,109.0,1,1989-10-24,36.347707,False,False,False
3,app_024,None,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000.0,...,0.0,True,None,575.0,1,1983-04-25,42.847365,False,False,False
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000.0,...,31763.0,False,algorithm_risk_score,463.0,1,1999-05-21,26.776181,False,False,False


## 10. Before vs After Validation (Re-running Quality Checks)

To demonstrate the impact of remediation, we re-run the same core data quality checks on both:
- the original dataset (`df`)
- the cleaned dataset (`df_clean`)

We encapsulate the checks in a reusable function (`build_issue_table`) to ensure:
- consistency of metrics across runs
- reproducibility of results
- easy extension of checks in future iterations

Finally, we merge the two issue tables to compare counts and percentages **before vs after cleaning**.

In [15]:
def build_issue_table(df_in: pd.DataFrame) -> pd.DataFrame:
    out = []
    # duplicates
    out.append(show_issue("Duplicate _id", "Consistency", df_in["_id"].duplicated(keep=False)))
    # incomplete key fields
    out.append(show_issue("Incomplete record (≥1 key field missing)", "Completeness", df_in[key_fields].isna().any(axis=1)))
    # non-numeric
    for col in ["annual_income","credit_history_months","debt_to_income","savings_balance"]:
        numeric = pd.to_numeric(df_in[col], errors="coerce")
        mask = df_in[col].notna() & numeric.isna()
        out.append(show_issue(f"{col} stored as non-numeric", "Consistency", mask))
    # invalid dob
    dob_p = df_in["date_of_birth"].apply(parse_date)
    out.append(show_issue("Invalid/unparseable date_of_birth", "Validity", dob_p.isna() & df_in["date_of_birth"].notna()))
    # invalid values
    ch = pd.to_numeric(df_in["credit_history_months"], errors="coerce")
    out.append(show_issue("Negative credit_history_months", "Validity", ch < 0))
    return pd.DataFrame(out)

before = build_issue_table(df)
after = build_issue_table(df_clean)

before.merge(after, on=["issue","dimension"], suffixes=("_before","_after"))

,issue,dimension,count_before,pct_before,count_after,pct_after
0,Duplicate _id,Consistency,4,0.8,0,0.0
1,Incomplete record (≥1 key field missing),Completeness,6,1.2,11,2.2
2,annual_income stored as non-numeric,Consistency,0,0.0,0,0.0
3,credit_history_months stored as non-numeric,Consistency,0,0.0,0,0.0
4,debt_to_income stored as non-numeric,Consistency,0,0.0,0,0.0
5,savings_balance stored as non-numeric,Consistency,0,0.0,0,0.0
6,Invalid/unparseable date_of_birth,Validity,4,0.8,4,0.8
7,Negative credit_history_months,Validity,2,0.4,0,0.0


In [16]:
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(out_dir / "credit_applications_clean.parquet", index=False)
df_clean.to_csv(out_dir / "credit_applications_clean.csv", index=False)

print("Saved:", (out_dir / "credit_applications_clean.parquet").resolve())

Saved: /Users/edoardosirianni/data/processed/credit_applications_clean.parquet


In [17]:
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

df_clean.to_csv(out_dir / "credit_applications_clean.csv", index=False)
print("Saved:", (out_dir / "credit_applications_clean.csv").resolve())

Saved: /Users/edoardosirianni/dego-project-team12/outputs/credit_applications_clean.csv


## 11. Overall Record-Level Quality Impact

In addition to analyzing individual issues separately, we compute a global indicator:

- A record is flagged if it has **at least one data quality issue**.
- We combine all previously defined issue masks into a single boolean matrix.
- We calculate:
  - the total number of affected records
  - the percentage of records impacted by at least one issue

This provides a high-level view of overall dataset reliability and quantifies the operational impact of data quality problems.

In [18]:
issue_masks = {
    "duplicate_id": dup_mask,
    "incomplete_record": incomplete_mask,
    "invalid_dob": invalid_dob_mask,
    "nonstandard_gender": nonstandard_mask,
    "neg_credit_history": neg_ch_mask,
    "neg_income": neg_inc_mask,
    "neg_savings": neg_sav_mask,
    "dti_out_of_range": dti_out_mask,
}

any_issue_mask = pd.DataFrame(issue_masks).fillna(False).any(axis=1)

any_issue_count = int(any_issue_mask.sum())
any_issue_pct = round(any_issue_count / len(df) * 100, 2)

any_issue_count, any_issue_pct

(18, 3.59)

## 12. Rubric-Ready Summary Table (Before vs After)

To align with the engineering deliverable requirements, we create a single consolidated table that:

- Lists the main data quality issues identified in the dataset
- Maps each issue to a **data quality dimension** (Completeness, Consistency, Validity, Accuracy)
- Quantifies impact **before and after** remediation using:
  - `count_before` / `pct_before`
  - `count_after` / `pct_after`

Implementation notes:
- `issue_row()` standardizes how counts and percentages are computed and stored.
- “After” masks are recomputed on `df_clean` to ensure the comparison is consistent and auditable.
- Gender is validated against the **expected final standardized set** (`Male`, `Female`, `Other/Unknown`).

This table is the primary evidence that remediation improved data quality.

In [19]:
def issue_row(issue, dimension, mask_before, mask_after=None):
    cb, pb = count_pct(mask_before)
    row = {"issue": issue, "dimension": dimension, "count_before": cb, "pct_before": pb}
    if mask_after is not None:
        ca, pa = count_pct(mask_after)
        row.update({"count_after": ca, "pct_after": pa})
    return row

# Ricrea le stesse metriche "after" su df_clean
dup_after = df_clean["_id"].duplicated(keep=False)
incomplete_after = df_clean[key_fields].isna().any(axis=1)

dob_parsed_after = df_clean["date_of_birth"].apply(parse_date)
invalid_dob_after = dob_parsed_after.isna() & df_clean["date_of_birth"].notna()

gender_raw_after = df_clean["gender"].dropna().astype(str).str.strip().str.lower()
allowed = {"male","female","m","f"}  # prima della normalizzazione
# Dopo la normalizzazione vuoi che siano Male/Female/Other/Unknown, quindi non serve questo check.
# Facciamo check "value not in expected final set":
expected_final = {"Male", "Female", "Other/Unknown"}
nonstandard_gender_after = df_clean["gender"].notna() & ~df_clean["gender"].isin(expected_final)

ch_after = pd.to_numeric(df_clean["credit_history_months"], errors="coerce")
neg_ch_after = ch_after < 0

inc_after = pd.to_numeric(df_clean["annual_income"], errors="coerce")
neg_inc_after = inc_after < 0

sav_after = pd.to_numeric(df_clean["savings_balance"], errors="coerce")
neg_sav_after = sav_after < 0

dti_after = pd.to_numeric(df_clean["debt_to_income"], errors="coerce")
dti_out_after = (dti_after < 0) | (dti_after > 1.5)

rubric_table = pd.DataFrame([
    issue_row("Duplicate records (_id)", "Consistency", dup_mask, dup_after),
    issue_row("Missing/incomplete records (≥1 key field missing)", "Completeness", incomplete_mask, incomplete_after),
    issue_row("Inconsistent gender coding", "Consistency", nonstandard_mask, nonstandard_gender_after),
    issue_row("Inconsistent/invalid date formats (DOB unparseable)", "Validity", invalid_dob_mask, invalid_dob_after),
    issue_row("Invalid values: negative credit_history_months", "Validity", neg_ch_mask, neg_ch_after),
    issue_row("Invalid values: negative annual_income", "Validity", neg_inc_mask, neg_inc_after),
    issue_row("Invalid values: negative savings_balance", "Validity", neg_sav_mask, neg_sav_after),
    issue_row("Invalid values: debt_to_income out of range", "Validity", dti_out_mask, dti_out_after),
]).sort_values("pct_before", ascending=False)

rubric_table

,issue,dimension,count_before,pct_before,count_after,pct_after
1,Missing/incomplete records (≥1 key field missing),Completeness,6,1.2,11,2.2
0,Duplicate records (_id),Consistency,4,0.8,0,0.0
3,Inconsistent/invalid date formats (DOB unparse...,Validity,4,0.8,4,0.8
2,Inconsistent gender coding,Consistency,2,0.4,0,0.0
4,Invalid values: negative credit_history_months,Validity,2,0.4,0,0.0
6,Invalid values: negative savings_balance,Validity,1,0.2,0,0.0
7,Invalid values: debt_to_income out of range,Validity,1,0.2,0,0.0
5,Invalid values: negative annual_income,Validity,0,0.0,0,0.0


In [20]:
rubric_table.to_csv("outputs/data_quality_rubric_table.csv", index=False)
print("Saved outputs/data_quality_rubric_table.csv")

Saved outputs/data_quality_rubric_table.csv
